In [1]:
import numpy as np
import os
import shutil
import struct
import subprocess
import tempfile
from collections.abc import Sequence
from pathlib import Path
import pandas as pd
import pyscf
import math
from pyscf import gto, scf, mcscf, cc, tools, lib, dmrgscf
from qiskit_addon_sqd.fermion import SCIResult, SCIState, bitstring_matrix_to_ci_strs
import ctypes
import platform
from tqdm.notebook import tqdm
dmrgscf.settings.BLOCKEXE = os.popen("which block2main").read().strip()
dmrgscf.settings.MPIPREFIX = ''
from classical import run_DMRG
# from classical import run_DMRG

In [2]:
input_template = """\
import os
import pandas as pd
from classical import run_DMRG

structure = {structure!r}
basis = {basis!r}
n_electrons = {n_electrons!r}
num_orbitals = {num_orbitals!r}
sym = {sym!r}
spin_sq = {spin_sq}
charge = {charge}
n_jobs = {n_jobs}
out_csv = "dmrg_Energy.csv"

energy = run_DMRG(structure, basis, n_electrons, num_orbitals, sym, spin_sq, charge, n_jobs=n_jobs)
df = pd.DataFrame([{{"energy": energy}}])
df.to_csv(out_csv, index=False)
print("Done writing dmrg_Energy.csv")
"""

slurm_template = """\
#!/bin/bash
#SBATCH --time=0-1:30:00
#SBATCH --account=rrg-jacobsen-ab
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=32
#SBATCH --mem-per-cpu=5GB
#SBATCH --job-name="{job_name}"
#SBATCH --error=job.e%J
#SBATCH --output=job.o%J

echo 'About to run python file'

# Load modules in correct order — StdEnv first
module load StdEnv/2023
module load python/3.11
module load openmpi
module load symengine rust
module load hdf5
module load openblas

echo "TEMP DIR: $SLURM_TMPDIR"

virtualenv --no-download $SLURM_TMPDIR/env
source $SLURM_TMPDIR/env/bin/activate

pip install --no-index --upgrade pip
pip install -e /home/gjones/projects/def-jacobsen/gjones/qiskit-addon-dice-solver/
pip install -e /home/gjones/scratch/distributed_LUCJ/
pip install git+https://github.com/pyscf/dmrgscf

PYSCFHOME=$(python -c "import pyscf; import os; print(os.path.dirname(pyscf.__file__))")
echo "PySCF home: $PYSCFHOME"

wget https://raw.githubusercontent.com/pyscf/dmrgscf/master/pyscf/dmrgscf/settings.py.example
mv settings.py.example ${{PYSCFHOME}}/dmrgscf/settings.py
chmod +x ${{PYSCFHOME}}/dmrgscf/nevpt2_mpi.py

pip install 'block2==0.5.3'

export LD_LIBRARY_PATH=$EBROOTOPENBLAS/lib:$LD_LIBRARY_PATH
export PATH=$SLURM_TMPDIR/env/bin:$PATH
export OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK
export OPENBLAS_NUM_THREADS=$SLURM_CPUS_PER_TASK
export MKL_NUM_THREADS=$SLURM_CPUS_PER_TASK
export NUMEXPR_NUM_THREADS=$SLURM_CPUS_PER_TASK

export LD_LIBRARY_PATH=$SLURM_TMPDIR/env/lib:$LD_LIBRARY_PATH

echo "OMP_NUM_THREADS=$OMP_NUM_THREADS"
echo "Running in directory: $(pwd)"
echo "Using input file: dmrg_input.py"
python dmrg_input.py
echo "File run complete for {job_name}."
"""



In [3]:
# # Root directory for generated jobs
# root = "diatomics"
# # -------------------------------------------------------------------------
# # Directory and file generation
# # -------------------------------------------------------------------------

# os.makedirs(root, exist_ok=True)

# for _, row in tqdm(diatomics_params.iterrows(), total=len(diatomics_params)):

#     rowdata = row.to_dict()

#     # Compute correct relative structure path
#     #
#     # You are generating:
#     #   diatomics/<basis>/<name>/dmrg_input.py
#     #
#     # From this directory to reach:
#     #   ../experiments/molecules/diatomics/XY.xyz
#     #
#     # We need:
#     #   ../../../../experiments/molecules/diatomics/XY.xyz
#     #
#     structure_rel = os.path.join(
#         "../../../../experiments",
#         rowdata["structure"]
#     )

#     basis = str(rowdata["basis"])
#     sym = str(rowdata["sym"])
#     spin_sq = int(rowdata["spin_sq"])
#     charge = int(rowdata["charge"])
#     n_froz = int(rowdata["n_froz"])
#     active_space = eval(rowdata["active_space"])

#     # e.g. "HN" from "molecules/diatomics/HN.xyz"
#     name = os.path.basename(rowdata["structure"]).replace(".xyz", "")

#     # Directory to create
#     job_dir = os.path.join(root, basis, name)
#     os.makedirs(job_dir, exist_ok=True)

#     # Write dmrg_input.py
#     with open(os.path.join(job_dir, "dmrg_input.py"), "w") as f:
#         f.write(input_template.format(
#             structure=structure_rel,
#             basis=basis,
#             active_space=active_space,
#             sym=sym,
#             spin_sq=spin_sq,
#             charge=charge,
#             n_froz=n_froz,
#             n_jobs=8
#         ))

#     # Write dmrg_run.sh
#     with open(os.path.join(job_dir, "dmrg_run.sh"), "w") as f:
#         f.write(slurm_template.format(
#             job_name=f"DMRG/{basis}/{name}"
#         ))

#     # Make dmrg_run.sh executable
#     os.chmod(os.path.join(job_dir, "dmrg_run.sh"), 0o755)


In [4]:
basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

active_spaces = pd.read_csv('../DDLUCJ_active_spaces_unfrozen.csv',delimiter=';').dropna(axis=1)


df=pd.read_csv('energies.csv',index_col=0)


In [5]:
active_spaces

,molecule,formula,xyz,No,Ne
0,ammonia,NH3,ammonia157.xyz,8,10
1,methane,CH4,methane50.xyz,9,10
2,ethylene,C2H4,ethylene42.xyz,14,16
3,ethane,C2H6,ethane28.xyz,16,18
4,water,H2O,water183.xyz,7,10
5,formaldehyde,CH2O,formaldehyde138.xyz,12,16
6,methanol,CH3OH,methanol22.xyz,14,18
7,fluoroform,CHF3,GDB04_5.xyz,21,34
8,"buta-1,3-diene",C4H6,GDB04_53.xyz,26,30
9,but-1-yne,C4H6,GDB04_49.xyz,26,30


In [7]:



# All molecules are uncharged and closed-shell
open_shell = False
spin_sq = 0
energies = {}

#Iterate over basis sets
for b in basis_sets:
    energies[b] = {}
    if os.path.exists(b) is False:
        os.mkdir(b)
    # Iterate over DataFrame rows and find the structures in the directory
    for row in list(active_spaces.itertuples(index=False)):
        structure_dict = row._asdict()
        subdir = structure_dict['molecule']
        molname = structure_dict['xyz']
        molelec = structure_dict['Ne']
        molorb = structure_dict['No']

        datadir = os.path.join(b,subdir)
        if os.path.exists(datadir) is False:
            os.mkdir(datadir)        
        # Find path
        structpath = f"../../structures/{molname}"
        # if os.path.exists(structpath):
        print(molname,b)
        # df = run_DMRG(structpath, b,molelec,molorb,None, 0,0,n_jobs=8)
        # energies[b][molname] = df
        with open(os.path.join(datadir, "dmrg_input.py"), "w") as f:
            f.write(input_template.format(
                structure=structpath,
                basis=b,
                n_electrons=molelec,
                num_orbitals=molorb,                    
                sym=None,
                spin_sq=0,
                charge=0,
                n_jobs=8
            ))
    
        # Write dmrg_run.sh
        with open(os.path.join(datadir, "dmrg_run.sh"), "w") as f:
            f.write(slurm_template.format(
                job_name=f"DMRG/{b}/{subdir}"
            ))
    
        # Make dmrg_run.sh executable
        os.chmod(os.path.join(datadir, "dmrg_run.sh"), 0o755)





# # Flatten the dictionary
# records = []
# for basis_set, molecules in energies.items():
#     for molecule, methods in molecules.items():
#         for method, energy in methods.items():
#             records.append((basis_set, molecule, method, energy))

# # Create DataFrame
# df = pd.DataFrame(records, columns=['Basis Set', 'Molecule', 'Method', 'Energy'])

# # Set MultiIndex
# # df.set_index(['Basis Set', 'Molecule', 'Method'], inplace=True)






# df.to_csv('energies.csv')









ammonia157.xyz STO-3G
methane50.xyz STO-3G
ethylene42.xyz STO-3G
ethane28.xyz STO-3G
water183.xyz STO-3G
formaldehyde138.xyz STO-3G
methanol22.xyz STO-3G
GDB04_5.xyz STO-3G
GDB04_53.xyz STO-3G
GDB04_49.xyz STO-3G
GDB04_33.xyz STO-3G
GDB04_65.xyz STO-3G
ammonia157.xyz cc-pVDZ
methane50.xyz cc-pVDZ
ethylene42.xyz cc-pVDZ
ethane28.xyz cc-pVDZ
water183.xyz cc-pVDZ
formaldehyde138.xyz cc-pVDZ
methanol22.xyz cc-pVDZ
GDB04_5.xyz cc-pVDZ
GDB04_53.xyz cc-pVDZ
GDB04_49.xyz cc-pVDZ
GDB04_33.xyz cc-pVDZ
GDB04_65.xyz cc-pVDZ
ammonia157.xyz aug-cc-pVDZ
methane50.xyz aug-cc-pVDZ
ethylene42.xyz aug-cc-pVDZ
ethane28.xyz aug-cc-pVDZ
water183.xyz aug-cc-pVDZ
formaldehyde138.xyz aug-cc-pVDZ
methanol22.xyz aug-cc-pVDZ
GDB04_5.xyz aug-cc-pVDZ
GDB04_53.xyz aug-cc-pVDZ
GDB04_49.xyz aug-cc-pVDZ
GDB04_33.xyz aug-cc-pVDZ
GDB04_65.xyz aug-cc-pVDZ
